# 05 — Pre-event Sentinel-2 acquisition (before Ditwah)

Strategy decisions (Stage 1 of the brief), written down here so they're graded:

| Decision | Choice | Why |
|---|---|---|
| Task framing | FORECASTING, not detection | Every pixel must predate the failure. Post-event imagery shows the scar itself — a model trained on it detects what already happened and cannot forecast anything. |
| Primary window | 2025-09-01 to 2025-11-20 | Slope condition immediately before the storm. Rain began ~24 Nov, landfall 28 Nov; the 4-day margin excludes early convective rainfall. |
| Fallback window | 2025-06-01 to 2025-08-31 | For locations the primary window can't clear. Still strictly pre-event. |
| Coordinates | all_coordinates.csv | Thinned positives (min 640 m separation) + matched negatives from a 1-4 km ring. Both classes, same window, same sensor — the only difference is what happened afterwards. |
| Patch footprint | 640 m x 640 m -> 64x64 px @ 10 m | Mapped slides average ~3,000 m² (~60 m across); 640 m gives roughly 10x context for slope and drainage. |
| Bands | B02, B03, B04, B08 | RGB + NIR, all 10 m native. NIR enables NDVI later. |
| Cloud filter | scene < 70%, then patch-level < 25% from SCL | A clear scene can still have your patch under cloud. The patch check is the one that matters. Relaxed from 20% because Sept-Nov is the NE monsoon onset. |
| SCL classes masked | 0,1,2,3,7,8,9,10,11 | Includes unclassified and dark-area, where thin cloud edges land. |
| Candidates | 8 per window | More attempts before giving up, since cloud rejection is higher pre-event. |
| Alignment | WarpedVRT to EPSG:32644 | Every patch pixel-identical despite different source tiles and orbits. |
| Resumability | progress CSV appended; 'success' rows skipped | Safe to stop and re-run at any point. |

Hard rule: no acquisition_date may exceed 2025-11-20. Verified in the final cell.

In [ ]:
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box

import rasterio
from rasterio.enums import Resampling
from rasterio.vrt import WarpedVRT
from rasterio.transform import from_bounds as transform_from_bounds
from rasterio.windows import from_bounds as window_from_bounds

from PIL import Image

import pystac_client
import planetary_computer

from tqdm.auto import tqdm

print("All imports OK")

In [ ]:
PROJECT_DIR = Path("..").resolve()
COORD_CSV = PROJECT_DIR / "data" / "coordinates" / "all_coordinates.csv"

TIF_DIR      = PROJECT_DIR / "data" / "pre_event" / "native_tif"
PNG_DIR      = PROJECT_DIR / "data" / "pre_event" / "png_256"
META_DIR     = PROJECT_DIR / "data" / "metadata"
PROGRESS_CSV = META_DIR / "pre_event_progress.csv"

for d in (TIF_DIR, PNG_DIR, META_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- decisions from the markdown cell above ---
PATCH_SIZE_M = 640
PATCH_PIXELS = 64
DST_CRS = "EPSG:32644"
BANDS = ["B02", "B03", "B04", "B08"]

PRIMARY_START,  PRIMARY_END  = "2025-09-01", "2025-11-20"
FALLBACK_START, FALLBACK_END = "2025-06-01", "2025-08-31"

SCENE_CLOUD_PREFILTER = 70
PATCH_CLOUD_MAX = 0.25
CANDIDATES_PER_WINDOW = 8

MAX_RETRIES = 3
RETRY_BASE_DELAY = 2

# nodata, saturated, dark area, cloud shadow, unclassified,
# cloud medium/high, cirrus, snow
CLOUD_SCL_VALUES = {0, 1, 2, 3, 7, 8, 9, 10, 11}

STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"

assert PRIMARY_END <= "2025-11-20" and FALLBACK_END <= "2025-11-20", \
    "Pre-event windows must close before the Ditwah rainfall began"

print(f"PRE-EVENT | patch {PATCH_SIZE_M} m / {PATCH_PIXELS} px")
print(f"coords    : {COORD_CSV.name}")
print(f"output    : {TIF_DIR.parent.name}/")
print(f"primary   : {PRIMARY_START} -> {PRIMARY_END}")
print(f"fallback  : {FALLBACK_START} -> {FALLBACK_END}")

In [ ]:
coords = pd.read_csv(COORD_CSV)
assert "label" in coords.columns, \
    "Wrong CSV — expected all_coordinates.csv with a label column"

print(f"{len(coords)} locations "
      f"({(coords.label == 1).sum()} positive, "
      f"{(coords.label == 0).sum()} negative)")

gdf = gpd.GeoDataFrame(
    coords,
    geometry=gpd.points_from_xy(coords.longitude, coords.latitude),
    crs="EPSG:4326",
)
gdf_utm = gdf.to_crs(DST_CRS)

half = PATCH_SIZE_M / 2
gdf_utm["minx"] = gdf_utm.geometry.x - half
gdf_utm["maxx"] = gdf_utm.geometry.x + half
gdf_utm["miny"] = gdf_utm.geometry.y - half
gdf_utm["maxy"] = gdf_utm.geometry.y + half

locations = gdf_utm[["landslide_id", "label",
                     "minx", "miny", "maxx", "maxy"]].to_dict("records")
print("\nexample:", locations[0])

In [ ]:
catalog = pystac_client.Client.open(
    STAC_URL,
    modifier=planetary_computer.sign_inplace,
)
print("Connected:", catalog.title)

In [ ]:
def search_candidates(bounds_utm, start_date, end_date,
                      max_scene_cloud=SCENE_CLOUD_PREFILTER,
                      limit=CANDIDATES_PER_WINDOW):
    """Search Sentinel-2 L2A scenes covering a UTM bbox, least-cloudy first."""
    bbox_wgs84 = (
        gpd.GeoSeries([box(*bounds_utm)], crs=DST_CRS)
        .to_crs("EPSG:4326")
        .total_bounds
    )
    for attempt in range(MAX_RETRIES):
        try:
            search = catalog.search(
                collections=["sentinel-2-l2a"],
                bbox=bbox_wgs84.tolist(),
                datetime=f"{start_date}/{end_date}",
                query={"eo:cloud_cover": {"lt": max_scene_cloud}},
            )
            items = list(search.item_collection())
            items.sort(key=lambda it: it.properties.get("eo:cloud_cover", 100))
            return items[:limit]
        except Exception as exc:
            wait = RETRY_BASE_DELAY * (2 ** attempt) + random.uniform(0, 1)
            print(f"  search retry {attempt + 1}/{MAX_RETRIES} "
                  f"after {exc!r}, waiting {wait:.1f}s")
            time.sleep(wait)
    return []

In [ ]:
def read_aligned_band(href, bounds_utm, out_pixels, resampling):
    with rasterio.open(href) as src:
        with WarpedVRT(src, crs=DST_CRS, resampling=resampling) as vrt:
            window = window_from_bounds(*bounds_utm, transform=vrt.transform)
            data = vrt.read(
                1,
                window=window,
                out_shape=(out_pixels, out_pixels),
                resampling=resampling,
            )
    return data


def patch_cloud_fraction(item, bounds_utm, out_pixels=PATCH_PIXELS):
    scl_href = item.assets["SCL"].href
    scl = read_aligned_band(scl_href, bounds_utm, out_pixels, Resampling.nearest)
    return float(np.isin(scl, list(CLOUD_SCL_VALUES)).mean())


def read_patch(item, bounds_utm, bands=BANDS, out_pixels=PATCH_PIXELS):
    stack = []
    for band in bands:
        href = item.assets[band].href
        data = read_aligned_band(href, bounds_utm, out_pixels, Resampling.bilinear)
        stack.append(data)
    return np.stack(stack, axis=0)      # (bands, H, W)

In [ ]:
def save_patch(landslide_id, arr, bounds_utm):
    transform = transform_from_bounds(*bounds_utm, arr.shape[2], arr.shape[1])
    tif_path = TIF_DIR / f"{landslide_id}.tif"
    with rasterio.open(
        tif_path, "w",
        driver="GTiff",
        height=arr.shape[1], width=arr.shape[2], count=arr.shape[0],
        dtype=arr.dtype, crs=DST_CRS, transform=transform,
        compress="deflate",
    ) as dst:
        dst.write(arr)
        dst.descriptions = tuple(BANDS)

    # quick-look preview only — the model trains on the native GeoTIFF,
    # not this 8-bit stretched PNG
    rgb = arr[[2, 1, 0], :, :].astype(np.float32) / 10000.0   # B04,B03,B02
    rgb = np.clip(rgb * 3.0, 0, 1)
    rgb_u8 = (rgb * 255).astype(np.uint8).transpose(1, 2, 0)
    png_path = PNG_DIR / f"{landslide_id}.png"
    Image.fromarray(rgb_u8).resize((256, 256), Image.BILINEAR).save(png_path)

    return str(tif_path), str(png_path)

In [ ]:
def acquire_location(loc):
    bounds_utm = (loc["minx"], loc["miny"], loc["maxx"], loc["maxy"])
    windows = [
        (PRIMARY_START, PRIMARY_END),
        (FALLBACK_START, FALLBACK_END),
    ]

    for start, end in windows:
        items = search_candidates(bounds_utm, start, end)

        for item in items:
            try:
                cloud_frac = patch_cloud_fraction(item, bounds_utm)
            except Exception:
                continue          # unreadable SCL, try the next candidate

            if cloud_frac > PATCH_CLOUD_MAX:
                continue

            arr = None
            for attempt in range(MAX_RETRIES):
                try:
                    arr = read_patch(item, bounds_utm)
                    break
                except Exception:
                    wait = RETRY_BASE_DELAY * (2 ** attempt) + random.uniform(0, 1)
                    time.sleep(wait)

            if arr is None or not np.any(arr):
                continue          # retries failed, or patch was blank/nodata

            tif_path, png_path = save_patch(loc["landslide_id"], arr, bounds_utm)
            return {
                "landslide_id": loc["landslide_id"],
                "label": loc["label"],
                "status": "success",
                "scene_id": item.id,
                "scene_cloud": item.properties.get("eo:cloud_cover"),
                "patch_cloud": round(cloud_frac * 100, 2),
                "acquisition_date": (item.properties.get("datetime") or "")[:10],
                "window_used": f"{start}/{end}",
                "tif_path": tif_path,
                "png_path": png_path,
                "error": "",
            }

    return {
        "landslide_id": loc["landslide_id"],
        "label": loc["label"],
        "status": "no_clear_scene",
        "scene_id": "", "scene_cloud": "", "patch_cloud": "",
        "acquisition_date": "", "window_used": "",
        "tif_path": "", "png_path": "", "error": "",
    }

In [ ]:
FIELDNAMES = ["landslide_id", "label", "status", "scene_id", "scene_cloud",
              "patch_cloud", "acquisition_date", "window_used",
              "tif_path", "png_path", "error"]


def load_done_ids():
    if not PROGRESS_CSV.exists():
        return set()
    done = pd.read_csv(PROGRESS_CSV)
    return set(done.loc[done.status == "success", "landslide_id"])


def append_result(result):
    write_header = not PROGRESS_CSV.exists()
    pd.DataFrame([result])[FIELDNAMES].to_csv(
        PROGRESS_CSV, mode="a", header=write_header, index=False
    )

In [ ]:
import csv
with open(PROGRESS_CSV, newline="", encoding="utf-8") as f:
    rows = list(csv.reader(f))

print("total lines:", len(rows))
print("\nline 2   :", rows[1])
print("line 1000:", rows[999])
print("line 1535:", rows[1534])
print("last     :", rows[-1])

In [ ]:
import csv

OLD_FIELDS = ["landslide_id", "status", "scene_id", "scene_cloud", "patch_cloud",
              "acquisition_date", "window_used", "tif_path", "png_path", "error"]

rows = []
with open(PROGRESS_CSV, newline="", encoding="utf-8") as f:
    for parts in csv.reader(f):
        if not parts or parts[0] == "landslide_id":
            continue
        if len(parts) == 11:
            rows.append(dict(zip(FIELDNAMES, parts)))
        elif len(parts) == 10:
            r = dict(zip(OLD_FIELDS, parts))
            r["label"] = ""
            rows.append(r)

fixed = pd.DataFrame(rows)[FIELDNAMES]
fixed["label"] = fixed.landslide_id.map(coords.set_index("landslide_id").label)
fixed = fixed.drop_duplicates("landslide_id", keep="last")

stray = fixed[fixed.label.isna()]
if len(stray):
    print(f"{len(stray)} IDs not in all_coordinates.csv — dropping")
    print(stray.landslide_id.head().tolist())
    fixed = fixed.dropna(subset=["label"])

fixed["label"] = fixed.label.astype(int)
fixed.to_csv(PROGRESS_CSV, index=False)

print(f"repaired: {len(fixed)} rows")
print(fixed.status.value_counts())
print(fixed.label.value_counts().rename({1: "positive", 0: "negative"}))

In [ ]:
# Run this FIRST. Sept-Nov is the NE monsoon onset, so cloud rejection is
# much higher than the post-event window. Check the rate before committing.
done_ids = load_done_ids()
todo = [l for l in locations if l["landslide_id"] not in done_ids]
print(f"{len(done_ids)} done, {len(todo)} remaining")

for loc in tqdm(todo[:200]):
    try:
        result = acquire_location(loc)
    except Exception as exc:
        result = {
            "landslide_id": loc["landslide_id"], "label": loc["label"],
            "status": "error", "scene_id": "", "scene_cloud": "",
            "patch_cloud": "", "acquisition_date": "", "window_used": "",
            "tif_path": "", "png_path": "", "error": repr(exc),
        }
    append_result(result)

trial = pd.read_csv(PROGRESS_CSV)
print(trial.status.value_counts())
print(f"\nsuccess rate: {(trial.status == 'success').mean():.1%}")

In [ ]:
import matplotlib.pyplot as plt

recent = pd.read_csv(PROGRESS_CSV).query("status == 'success'").tail(12)
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for ax, (_, r) in zip(axes.ravel(), recent.iterrows()):
    ax.imshow(Image.open(r.png_path))
    ax.set_title(f"{r.landslide_id}\n{r.acquisition_date}  cl {r.patch_cloud}%",
                 fontsize=7)
    ax.axis("off")
plt.tight_layout(); plt.show()

print("latest acquisition:", recent.acquisition_date.max())

In [ ]:
print("in all_coordinates:", len(coords))
print("  positives:", (coords.label == 1).sum())
print("  negatives:", (coords.label == 0).sum())
print("\ndownloaded so far:", len(fixed))
print(fixed.label.value_counts())

In [ ]:
# Only run once the trial success rate looks acceptable (>= 70%).
# Resumable — safe to interrupt and re-run.
done_ids = load_done_ids()
todo = [l for l in locations if l["landslide_id"] not in done_ids]
print(f"{len(done_ids)} done, {len(todo)} remaining")

for loc in tqdm(todo):
    try:
        result = acquire_location(loc)
    except Exception as exc:
        result = {
            "landslide_id": loc["landslide_id"], "label": loc["label"],
            "status": "error", "scene_id": "", "scene_cloud": "",
            "patch_cloud": "", "acquisition_date": "", "window_used": "",
            "tif_path": "", "png_path": "", "error": repr(exc),
        }
    append_result(result)

In [ ]:
progress = pd.read_csv(PROGRESS_CSV)
ok = progress.query("status == 'success'")

print(progress.status.value_counts())
print(f"\noverall success rate: {(progress.status == 'success').mean():.1%}")

print(f"\n{len(ok)} patches downloaded")
print(ok.label.value_counts().rename({1: "positive", 0: "negative"}))

print("\nsuccess rate by class:")
print(progress.groupby("label").status
      .apply(lambda s: (s == "success").mean()).round(3))

print(f"\nacquisition dates: {ok.acquisition_date.min()} -> "
      f"{ok.acquisition_date.max()}")
assert ok.acquisition_date.max() <= "2025-11-20", \
    "LEAKAGE — post-event imagery in the training set"
print("leakage check PASSED — all imagery predates the Ditwah rainfall")

print(f"\nwindow usage:\n{ok.window_used.value_counts()}")
print(f"\ntop acquisition dates:\n{ok.acquisition_date.value_counts().head(5)}")